# LuyDek: train the "item on the mat" detector

Fine-tunes **YOLO26n-seg** (Ultralytics) to find *any product on the shop's mat*: one class, with its
outline. It doesn't name products: CLIP and the shop's photos still do that, so adding a product never
needs retraining. The training scenes are generated from the kit: the shop's products, cut out of their
own photos, laid on photos of the shop's own empty mat, touching, overlapping and in changing light.

**Run it:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. It asks for the kit zip
(`training/item-detector-kit.zip` on the LuyDek computer) and, at the end, downloads `item-detector.pt`.
Put that file in LuyDek's `models/` folder and tell Claude it's there: it gets compared with the current
detector on real counter photos before it is used.

Ultralytics YOLO is AGPL-3.0, like the detector LuyDek uses today.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU, then run all again"

## 1. The kit

In [ ]:
import io, zipfile
from google.colab import files
uploaded = files.upload()  # choose item-detector-kit.zip
zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))).extractall(".")
!ls kit; ls kit/cutouts | wc -l; ls kit/backgrounds | wc -l

In [ ]:
!pip -q install ultralytics==8.4.171

## 2. Training scenes

3,000 for training and 400 for checking (made from mat photos and product photos the training never sees).

In [ ]:
!python kit/generate.py --kit kit --out dataset --train 3000 --val 400 --seed 0

In [ ]:
# A look at a few scenes with their outlines, to check they look like the counter
import glob, cv2, numpy as np, matplotlib.pyplot as plt
paths = sorted(glob.glob("dataset/images/train/*.jpg"))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axes.flat, paths, strict=False):
    img = cv2.imread(p)
    h, w = img.shape[:2]
    for line in open(p.replace("images", "labels").replace(".jpg", ".txt")).read().split(chr(10)):
        if line.strip():
            pts = (np.array(line.split()[1:], float).reshape(-1, 2) * (w, h)).astype(np.int32)
            cv2.polylines(img, [pts], True, (0, 255, 0), max(2, w // 400))
    ax.imshow(img[..., ::-1]); ax.axis("off")
plt.tight_layout()

## 3. Train

About an hour on a T4 (it stops early when it stops improving).

In [ ]:
from ultralytics import YOLO, settings
settings.update({"sync": False})  # no usage analytics
model = YOLO("yolo26n-seg.pt")    # the pretrained starting point (downloaded by Ultralytics)
model.train(data="dataset/dataset.yaml", imgsz=640, epochs=60, batch=32, patience=15,
            project="runs", name="item", exist_ok=True, plots=True)

## 4. Check

In [ ]:
best = YOLO("runs/item/weights/best.pt")
metrics = best.val(data="dataset/dataset.yaml", imgsz=640)
print(f"outline mAP50-95: {metrics.seg.map:.3f}   box mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# Predictions on a few check scenes
paths = sorted(glob.glob("dataset/images/val/*.jpg"))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, r in zip(axes.flat, best.predict(paths, imgsz=640, verbose=False), strict=False):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
plt.tight_layout()

## 5. Download the model

In [ ]:
import shutil
shutil.copy("runs/item/weights/best.pt", "item-detector.pt")
files.download("item-detector.pt")